# **EDA**

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("echantillon")

In [0]:
print(df.info())
print(df.head())

In [0]:
# valeurs manquantes par colonne
valeurs_nulles = df.isnull().sum()
taux_valeurs_nulles = round((df.isnull().sum() / len(df)) * 100, 2)

df_nulles = pd.DataFrame({
    'Colonne': valeurs_nulles.index,
    'Nb_Manquants': valeurs_nulles.values,
    'Pourcentage': taux_valeurs_nulles.values
}).sort_values('Nb_Manquants', ascending=False)

print(df_nulles[df_nulles['Nb_Manquants'] > 0])

In [0]:
unique_rate = df["RatecodeID"].unique()
print(unique_rate)
unique_store = df["store_and_fwd_flag"].unique()
print(unique_store)
unique_surcharge = df["congestion_surcharge"].unique()
print(unique_surcharge)
unique_passenger = df["passenger_count"].unique()
print(unique_passenger)
unique_airport = df["airport_fee"].unique()
print(unique_airport)
unique_Airport=df["Airport_fee"].unique()
print(unique_Airport)



In [0]:
df["tpep_pickup_datetime"] = pd.to_datetime( df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
df["tpep_pickup_datetime"].dtype
df["tpep_dropoff_datetime"].dtype

In [0]:
#Vérifier les dates de debut vs fin 
(df["tpep_pickup_datetime"] > df["tpep_dropoff_datetime"]).any()

In [0]:
(df["tpep_pickup_datetime"] > df["tpep_dropoff_datetime"]).sum()

In [0]:
invalid_dates = df[df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]]
print(invalid_dates)

In [0]:
numeric_cols = df.select_dtypes(include="number").columns

cols_with_negatives = (
    df[numeric_cols]
    .lt(0)
    .any()
)

cols_with_negatives = cols_with_negatives[cols_with_negatives].index.tolist()

print(cols_with_negatives)

# **NETTOYAGE**

In [0]:
#Supprimer les doublons
df = df.drop_duplicates()

In [0]:
#Renverser les dates de debut et fin
mask = df["tpep_pickup_datetime"] > df["tpep_dropoff_datetime"]
df.loc[mask, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]] = (
    df.loc[mask, ["tpep_dropoff_datetime", "tpep_pickup_datetime"]].values
)


In [0]:
#fusionner les deux colonnes
df["airport_fee"]=df["airport_fee"].fillna(df["Airport_fee"])

In [0]:
#Supprimer les colonnes Airport_fee
df.drop(["Airport_fee"], axis=1, inplace=True)


In [0]:
df["airport_fee"]=df["airport_fee"].fillna(0.0)

In [0]:
df["passenger_count"]=df["passenger_count"].fillna(1.0)


In [0]:
som = (df["RatecodeID"] == 99).sum()
print(som)


In [0]:

df["RatecodeID"] = df["RatecodeID"].replace(99, pd.NA)
df["RatecodeID"] = df["RatecodeID"].fillna(0).astype(int)

In [0]:
# df["RatecodeID"] = (
#     df["RatecodeID"]
#     .replace(99, pd.NA)
#     .fillna(0)
#     .infer_objects(copy=False)
#     .astype(int)
# )

In [0]:
df["store_and_fwd_flag"]=df["store_and_fwd_flag"].fillna("N")


In [0]:
df["congestion_surcharge"]=df["congestion_surcharge"].fillna(0.0)

In [0]:
#Vérification 


# valeurs manquantes par colonne
valeurs_nulles = df.isnull().sum()
taux_valeurs_nulles = round((df.isnull().sum() / len(df)) * 100, 2)

df_nulles = pd.DataFrame({
    'Colonne': valeurs_nulles.index,
    'Nb_Manquants': valeurs_nulles.values,
    'Pourcentage': taux_valeurs_nulles.values
}).sort_values('Nb_Manquants', ascending=False)

print(df_nulles[df_nulles['Nb_Manquants'] > 0])

unique_rate = df["RatecodeID"].unique()
print(unique_rate)
unique_store = df["store_and_fwd_flag"].unique()
print(unique_store)
unique_surcharge = df["congestion_surcharge"].unique()
print(unique_surcharge)
unique_passenger = df["passenger_count"].unique()
print(unique_passenger)
unique_airport = df["airport_fee"].unique()
print(unique_airport)



# **Statistiques sur l'échantillon**

### Prix moyen d’une course (fare_amount) : Comparer l’estimation du sample vs la valeur exacte sur population

In [0]:
#Prix moyen d’une course
prix_moyen_course = df["fare_amount"].mean()
print(prix_moyen_course)

In [0]:
#Généralisation sur pop :
std  = df["fare_amount"].std()
n= len(df)
Standard_Error = std/ (n ** 0.5)


In [0]:
from scipy import stats

In [0]:
ci_low, ci_high = stats.t.interval(
    0.95,              # confidence level 95%
    df=n-1,            # degrees of freedom
    loc=prix_moyen_course,        # mean sample
    scale=Standard_Error           # standard error
)

print("Mean estimate:", prix_moyen_course)
print("95% CI:", ci_low, "-", ci_high)

###  Distance moyenne d’une course (trip_distance) : Identifier si l’échantillon est représentatif

In [0]:
#Distance moyenne d’une course
distance_moyenne_course = df["trip_distance"].mean()
print(distance_moyenne_course)

In [0]:
#Généralisation sur pop :
std1  = df["trip_distance"].std()
Standard_Error1 = std1/ (n ** 0.5)


In [0]:
ci_low1, ci_high1 = stats.t.interval(
    0.95,              # confidence level 95%
    df=n-1,            # degrees of freedom
    loc=distance_moyenne_course,        # mean sample
    scale=Standard_Error1           # standard error
)

print("Mean estimate:", distance_moyenne_course)
print("95% CI:", ci_low1, "-", ci_high1)

### Durée moyenne des courses : Calculer avec pickup/dropoff datetime

In [0]:
df["duree"] = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
df["duree_secondes"] = df["duree"].dt.total_seconds()
duree_moyenne_course_secondes = df["duree_secondes"].mean()
print(duree_moyenne_course_secondes) 

In [0]:
#Généralisation sur pop :
std2  = df["duree_secondes"].std()
Standard_Error2 = std2/ (n ** 0.5)

In [0]:
ci_low2, ci_high2 = stats.t.interval(
    0.95,              # confidence level 95%
    df=n-1,            # degrees of freedom
    loc=duree_moyenne_course_secondes,        # mean sample
    scale=Standard_Error2           # standard error
)

print("Mean estimate:", duree_moyenne_course_secondes)
print("95% CI:", ci_low2, "-", ci_high2)

#Conversion en minutes
duree_moyenne_course_minutes = duree_moyenne_course_secondes/60
ci_low_minutes = ci_low2/60
ci_high_minutes = ci_high2/60
print("Mean estimate:", duree_moyenne_course_minutes)
print("95% CI:", ci_low_minutes, "-", ci_high_minutes)


### Proportion des courses avec tip > 0 : Inférence vs valeur réelle 

In [0]:
#Proportion des courses avec tip > 0
prop_moyenne = (df["tip_amount"] > 0).mean()
print(prop_moyenne)


In [0]:
#Généralisation sur pop :
import numpy as np
Standard_Error3 = np.sqrt(prop_moyenne * (1 - prop_moyenne) / n)

In [0]:
ci_low3, ci_high3 = stats.norm.interval(
    0.95,              # confidence level 95%
    loc=prop_moyenne,        # mean sample
    scale=Standard_Error3           # standard error
)

print("Mean estimate:", prop_moyenne)
print("Poucentage de la moyenne est : ", prop_moyenne*100, "%")
print("95% CI:", ci_low3, "-", ci_high3)
print("pourcentage de l'intervalle:", ci_high3*100, "%", ci_low3*100, "%")

### Distribution des courses par heure/jour/semaine Identifier les heures de pointe

In [0]:
#par heure
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
#par jour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.day_name()
#par semaine
df["pickup_week"] = df["tpep_pickup_datetime"].dt.isocalendar().week

In [0]:
#Distribution des courses par heure
courses_par_heure = df.groupby("pickup_hour").size()
print(courses_par_heure)

#Graphe de distrubution des courses par heure
import matplotlib.pyplot as plt
plt.figure()
plt.plot(courses_par_heure.index, courses_par_heure.values)
plt.xlabel("Heure de la journée")
plt.ylabel("Nombre de courses")
plt.title("Distribution des courses par heure")
plt.show()

In [0]:
#Distribution des courses par jour
courses_par_jour = df.groupby("pickup_day").size()
print(courses_par_jour)

#Graphe de distrubution des courses par jour

plt.figure()
plt.plot(courses_par_jour.index, courses_par_jour.values)
plt.xlabel("Jour de la semaine")
plt.ylabel("Nombre de courses")
plt.title("Distribution des courses par jour")
plt.show()

In [0]:
#Distribution des courses par jour de la semaine
courses_par_semaine = df.groupby("pickup_week").size()
print(courses_par_semaine)

#Graphe de distrubution des courses par jour

plt.figure()
plt.plot(courses_par_semaine.index, courses_par_semaine.values)
plt.xlabel("Semaine du mois")
plt.ylabel("Nombre de courses")
plt.title("Distribution des courses par semaine")
plt.show()

### Comparaison des fares selon zones géographiques (pickup/dropoff boroughs) : Identifier si l’échantillon reflète la diversité spatiale

In [0]:
df[["PULocationID","DOLocationID"]]

In [0]:
df.groupby(["PULocationID", "DOLocationID"])["fare_amount"] \
  .agg(mean_fare="mean", nb_courses="size") \
  .sort_values("mean_fare", ascending=False) \
  .head(10)

In [0]:
prix_par_zone_depart = (
    df.groupby("PULocationID")["fare_amount"]
    .mean()
    .sort_values(ascending=False)
)

#Graphe des grands fares par zone de depart
top_zones = prix_par_zone_depart.head(10)

plt.figure()
plt.bar(top_zones.index.astype(str), top_zones.values)
plt.xlabel("Pickup LocationID")
plt.ylabel("Prix moyen")
plt.title("Top 10 zones (pickup) avec les fares les plus élevés")
plt.show()

In [0]:
prix_par_zone_arrive = (
    df.groupby("DOLocationID")["fare_amount"]
    .mean()
    .sort_values(ascending=False)
)

#Graphe des grands fares par zone de depart
top_zones_arr = prix_par_zone_arrive.head(10)

plt.figure()
plt.bar(top_zones_arr.index.astype(str), top_zones_arr.values)
plt.xlabel("Dropoff LocationID")
plt.ylabel("Prix moyen")
plt.title("Top 10 zones (dropoff) avec les fares les plus élevés")
plt.show()

### Analyse des outliers : Courses très longues ou très chères : impact sur estimation vs population

**Courses très chère**

In [0]:
#somme des courses très chères 
num = len((df[df["fare_amount"] > 200]["fare_amount"]))
print(num)



In [0]:
#moyenne des courses très chers
course_chers_moyenne =(df[df["fare_amount"] > 200]["fare_amount"]).mean()
print(course_chers_moyenne)

In [0]:
#Généralisation sur pop :
std5  = (df[df["fare_amount"] > 200]["fare_amount"]).std()
Standard_Error5 = std5/ (num ** 0.5)

In [0]:
ci_low5, ci_high5 = stats.t.interval(
    0.95,              # confidence level 95%
    df=num-1,            # degrees of freedom
    loc=course_chers_moyenne,        # mean sample
    scale=Standard_Error5           # standard error
)

print("Mean estimate:", course_chers_moyenne)
print("95% CI:", ci_low5, "-", ci_high5)


**Courses très longues**

In [0]:
#somme des courses très longues 
num2 = len((df[df["trip_distance"] > 50]["trip_distance"]))
print(num2)

In [0]:
#moyenne des courses très longues
course_dist_moyenne =(df[df["trip_distance"] > 50]["trip_distance"]).mean()
print(course_dist_moyenne)

In [0]:
#Généralisation sur pop :
std6  = (df[df["trip_distance"] > 50]["trip_distance"]).std()
Standard_Error6 = std6/ (num2 ** 0.5)

In [0]:
ci_low6, ci_high6 = stats.t.interval(
    0.95,              # confidence level 95%
    df=num2-1,            # degrees of freedom
    loc=course_dist_moyenne,        # mean sample
    scale=Standard_Error6           # standard error
)

print("Mean estimate:", course_dist_moyenne)
print("95% CI:", ci_low6, "-", ci_high6)

### Ratio tip/fare moyen par type de paiement (cash vs card)

In [0]:
df["payment_type"].value_counts()


**Ratio tip/fare moyen par type de paiement cash**

In [0]:
ratio_cash = (
    df.loc[
        (df["payment_type"] == 2) & (df["fare_amount"] > 0),
        "tip_amount"
    ] / 
    df.loc[
        (df["payment_type"] == 2) & (df["fare_amount"] > 0),
        "fare_amount"
    ]
)

In [0]:
ratio_cash_moy = ratio_cash.mean()
print("Cash:", ratio_cash_moy)


In [0]:
n_cash = ratio_cash.dropna().shape[0]
se_cash = ratio_cash.std(ddof=1) / np.sqrt(n_cash)

ci_cash = stats.t.interval(
    0.95,
    df=n_cash-1,
    loc=ratio_cash_moy,
    scale=se_cash
)

print(f"Cash 95% CI:{ ci_cash[0]:.9f} - {ci_cash[1]:.9f}")

**Ratio tip/fare moyen par type de paiement card**

In [0]:
ratio_card = (
    df.loc[
        (df["payment_type"] == 1) & (df["fare_amount"] > 0),
        "tip_amount"
    ] / 
    df.loc[
        (df["payment_type"] == 1) & (df["fare_amount"] > 0),
        "fare_amount"
    ]
)

In [0]:
ratio_card_moy = ratio_card.mean()
print("Card:", ratio_card_moy)

In [0]:
n_card = ratio_card.dropna().shape[0]
se_card = ratio_card.std(ddof=1) / np.sqrt(n_cash)

ci_card = stats.t.interval(
    0.95,
    df=n_card-1,
    loc=ratio_card_moy,
    scale=se_card
)

print(f"Cash 95% CI:{ ci_card[0]:.9f} - {ci_card[1]:.9f}")